# Auditar dentro de un nodo

`gradient_audit()` a nivel de grafo trata cada filtro como opaco: un
registro por nodo y paso. Con `inside=` los hooks entran en el modelo
del nodo y cada submódulo audita bajo un id jerárquico
(`encoder/<ruta.del.módulo>`): mismos registros, flags, persistencia y
figuras. Aquí lo vemos cazar un *vanishing gradient* de libro.

In [ ]:
import torch
import torch.nn as nn

import soma
from soma import AuditScope, DifferentiableFilter, Graph


class DeepEncoder(DifferentiableFilter):
    def __init__(self, layers=12):
        super().__init__(layers=layers)

    def build_module(self, input_shape):
        mods = []
        for _ in range(self.layers):
            lin = nn.Linear(input_shape[-1], input_shape[-1])
            with torch.no_grad():
                lin.weight.mul_(0.35)  # init contractiva: gradientes se apagan
            mods += [lin, nn.Tanh()]
        return nn.Sequential(*mods)

    def output_shape(self, input_shape):
        return input_shape


torch.manual_seed(0)
g = Graph()
g.node("encoder", DeepEncoder())
x, y = torch.randn(64, 8), torch.randn(64, 8)
g.materialize(x)
g.train()
g.make_optimizer(torch.optim.Adam, lr=1e-3)

## `inside=True`: cero configuración

Autoselección: los hijos directos con parámetros (los wrappers de un
solo hijo se descienden). El nodo raíz sigue auditándose bajo su id
plano. Alternativas, todas equivalentes por dentro:

```python
g.gradient_audit(inside={"encoder": 2})            # int = profundidad
g.gradient_audit(inside={"encoder": ["0", "4.*"]}) # patrones fnmatch
g.gradient_audit(inside={"encoder": AuditScope(depth=3, sample_every=10)})

class DeepEncoder(DifferentiableFilter):
    _audit_scope = "auto"   # declarado donde vive el modelo
```

Precedencia: `inside[...]` > `_audit_scope` de la clase > auto.

In [ ]:
with g.track_run("nb08-inside", tags=["demo"]) as run:
    with g.gradient_audit(inside=True) as audit:
        for epoch in range(4):
            with g.context() as ctx:
                g.zero_grad()
                out, _ = g.forward(x)
                g.backward(ctx, nn.functional.mse_loss(out, y))
            g.step(ctx)

print(audit.report().pretty())

## El flujo de gradientes, capa a capa

El gráfico clásico: submódulos en orden de ejecución real en x, `|∂|`
en escala log en y. Un *vanishing* se lee como escalera que cae hacia
la entrada — aquí, varios órdenes de magnitud.

In [ ]:
view = soma.RunView(run.dir)
view.plot_module_flow("encoder")

## La arquitectura interna, anotada

El árbol de submódulos se persiste en `diagnostics/modules/encoder.json`
(mismo esquema Graph que `graph.json`) y se renderiza con el mismo
mecanismo de overlay: parámetros, `|∂|` medio y flags por capa.

In [ ]:
print(view.to_mermaid(node="encoder"))

## Flags con rollup

Cada capa flaggeada emite su `HealthFlag` propio **y** el nodo padre
recibe uno agregado por familia cuyo detalle nombra las capas — el DAG
externo marca el nodo, las vistas internas señalan la capa exacta.

In [ ]:
parent = [f for f in view.health_flags() if f["node_id"] == "encoder"]
children = [f for f in view.health_flags() if f["node_id"].startswith("encoder/")]
print(f"{len(children)} flags de capa; rollup en el padre:")
[f["detail"] for f in parent]

In [ ]:
# Series temporales por capa (por defecto plot_audit solo muestra raíces).
view.plot_audit(node="encoder", metric="out_grad.norm")

`soma report <run_id>` incluye todo esto en la sección **Module flow**
del informe. Para modelos grandes: `AuditScope(sample_every=N)` muestrea
los submódulos cada N pasos (el raíz siempre registra), y `max_modules`
limita la selección avisando — nunca en silencio.